# Model Deployment

A practical reference for taking a trained model from an artifact in object storage to a reliable, observable, and scalable production service. Covers serving patterns (online, batch, streaming, edge), packaging, model servers (TorchServe, Triton, KServe, vLLM), containers and Kubernetes, autoscaling, rollout strategies, and monitoring.

## Table of Contents

1. [Introduction](#introduction)
2. [Key Features](#key-features)
3. [Architecture Overview](#architecture)
4. [Installation](#installation)
5. [Basic Usage](#basic-usage)
6. [Advanced Features](#advanced-features)
7. [Use Cases](#use-cases)
8. [Best Practices](#best-practices)
9. [Common Pitfalls](#pitfalls)
10. [Performance Optimization](#performance)
11. [Production Deployment](#deployment)
12. [Monitoring and Observability](#monitoring)
13. [Troubleshooting](#troubleshooting)
14. [Comparison with Alternatives](#comparison)
15. [Resources](#resources)

## Introduction

**Model deployment** is the set of practices that turn a trained model artifact into a running service that other systems can call to get predictions. Training produces a file (weights + config); deployment is everything that happens after: packaging the model with its inference code, exposing it behind an API or batch pipeline, sizing and scaling the compute, rolling out new versions safely, and watching it in production.

### What is it?

Deployment is the bridge between data science and a live product. A model that scores 0.95 AUC in a notebook delivers zero value until a request can reach it, get a prediction within the latency budget, and have that prediction logged for monitoring. The discipline spans four common serving modes:

- **Online / real-time** — synchronous request/response behind HTTP or gRPC (e.g. fraud scoring on checkout). Latency-sensitive, often single-digit to low-hundreds of milliseconds.
- **Batch** — score a large dataset on a schedule and write results to a table or file (e.g. nightly churn scores). Throughput-sensitive, latency irrelevant.
- **Streaming** — score events as they arrive from a queue (Kafka/Kinesis) with at-least-once or exactly-once semantics.
- **Edge / on-device** — run the model on a phone, browser, or IoT device with no network round trip (ONNX Runtime, TensorFlow Lite, Core ML).

### Why use it?

- **Decouples the model lifecycle from the application.** The app calls a stable API; the model can be retrained and redeployed without an app release.
- **Reproducibility.** A versioned container pins the model, framework, CUDA, and preprocessing together, eliminating "works on my machine."
- **Scalability and isolation.** Inference scales independently of the rest of the system, and a heavy model cannot starve unrelated services.
- **Safe iteration.** Canary, blue-green, and shadow deployments let you ship new model versions while bounding blast radius.

### When to use it?

- A model needs to serve predictions to a product, partner, or internal tool on demand.
- You need to score data on a schedule at scale and persist the results.
- You need to compare a candidate model against production traffic (shadow / A-B) before promoting it.
- Predictions must run where there is no reliable network connection (mobile, embedded, browser).

## Key Features

### Core Capabilities of a Model Deployment Stack

| Feature | Description | Benefit |
|---------|-------------|---------|
| Standardized model packaging | Bundle weights, inference handler, and dependencies (model archive or container image) | Reproducible, portable deploys; no environment drift |
| Multi-framework serving | One server runs PyTorch, TensorFlow, ONNX, XGBoost, etc. (Triton, KServe) | Consolidate infra; uniform ops across teams |
| Dynamic / adaptive batching | Server groups concurrent requests into one forward pass | Higher GPU utilization and throughput at a small latency cost |
| Autoscaling | Scale replicas (and scale-to-zero) on QPS, latency, or GPU/CPU load | Match cost to demand; survive traffic spikes |
| Versioning + rollout control | Multiple model versions live behind canary / blue-green / shadow routing | Ship safely; instant rollback on regression |
| Observability hooks | Latency, throughput, error, saturation metrics + prediction/feature logging | Detect drift, regressions, and incidents fast |
| Hardware acceleration | GPU/TPU/Inferentia support, FP16/INT8, TensorRT, vLLM paged attention | Lower latency and cost per prediction |

## Architecture Overview

A typical online-serving architecture, from client to model:

```
                       +---------------------+
  client ── HTTPS ──►  |  API gateway / LB   |  (TLS, authn, rate limit)
                       +----------+----------+
                                  │
                                  ▼
                       +---------------------+
                       | Ingress / Service   |  (Kubernetes)
                       +----------+----------+
                                  │  (round-robin / least-conn)
                  ┌───────────────┼───────────────┐
                  ▼               ▼               ▼
            +-----------+   +-----------+   +-----------+
            | Inference |   | Inference |   | Inference |   ◄── HPA / KEDA autoscaler
            |  pod (v2) |   |  pod (v2) |   |  pod (v1) |   ◄── canary: 90% v2 / 10% v1
            +-----+-----+   +-----+-----+   +-----+-----+
                  │  model server (Triton / TorchServe / vLLM)
                  ▼
          +-------------------+         +---------------------+
          | Model registry /  |         | Feature store /     |
          | object store (S3) |         | online cache (Redis)|
          +-------------------+         +---------------------+

  side channels:  Prometheus (metrics) · OpenTelemetry (traces) · prediction log → S3/BigQuery (drift)
```

### Components

1. **API gateway / load balancer** — terminates TLS, authenticates callers, enforces rate limits, and spreads traffic across replicas.
2. **Inference replicas** — pods running a model server that loads weights from the registry and exposes HTTP/gRPC inference endpoints plus health and metrics endpoints.
3. **Model registry / artifact store** — versioned source of truth for model files (MLflow Model Registry, S3/GCS, or a model server's repository layout).
4. **Feature store / cache** — supplies real-time features and caches hot lookups so inference is not blocked on upstream databases.
5. **Autoscaler** — Horizontal Pod Autoscaler or KEDA adds/removes replicas based on QPS, latency, or queue depth.
6. **Observability plane** — Prometheus/Grafana for metrics, OpenTelemetry for traces, and an asynchronous prediction log for drift and audit.

## Installation

### Prerequisites

- Python 3.9+
- Docker (to build and run container images)
- A model server, depending on framework: `torchserve`, NVIDIA Triton, or `vllm` for LLMs
- For Kubernetes deploys: `kubectl` and a cluster (kind/minikube locally, or EKS/GKE/AKS)
- CUDA toolkit + an NVIDIA GPU for accelerated serving (optional for CPU models)

### Installation Steps

The examples below use FastAPI for a minimal custom server, plus optional model servers. Uncomment the cell to install in Colab or a fresh environment.

In [ ]:
# Uncomment to install the serving dependencies used in this notebook.
# Minimal custom server:
# !pip install fastapi "uvicorn[standard]" pydantic scikit-learn joblib
#
# Framework model servers (choose what matches your model):
# !pip install torchserve torch-model-archiver    # PyTorch
# !pip install tritonclient[all]                   # NVIDIA Triton client
# !pip install vllm                                # LLM serving (GPU)
# !pip install bentoml                             # framework-agnostic packaging
print("Install the lines that match your stack, then restart the kernel.")

## Basic Usage

### Quick Start Example

The smallest useful deployment is a model wrapped in an HTTP API. Below we train a tiny scikit-learn classifier, persist it, and serve it with FastAPI. This is the pattern every framework formalizes: **load once at startup, predict per request, validate inputs, return JSON.**

In [ ]:
# 1) Train and persist a model artifact (done once, offline).
import joblib
from sklearn.datasets import load_iris
from sklearn.ensemble import RandomForestClassifier

X, y = load_iris(return_X_y=True)
clf = RandomForestClassifier(n_estimators=50, random_state=0).fit(X, y)
joblib.dump(clf, "/tmp/iris_model.joblib")
print("Saved model artifact -> /tmp/iris_model.joblib")

In [ ]:
# 2) A minimal FastAPI serving app (normally its own app.py).
# Key ideas: load the model ONCE at import/startup, validate the request
# schema, and expose a /healthz probe separate from the predict path.
serving_app = '''
from fastapi import FastAPI
from pydantic import BaseModel, conlist
import joblib

app = FastAPI(title="iris-classifier", version="1.0.0")
model = joblib.load("/tmp/iris_model.joblib")  # loaded once, reused per request

class PredictRequest(BaseModel):
    instances: list[conlist(float, min_length=4, max_length=4)]

class PredictResponse(BaseModel):
    predictions: list[int]

@app.get("/healthz")          # liveness/readiness probe — no model work
def healthz():
    return {"status": "ok"}

@app.post("/predict", response_model=PredictResponse)
def predict(req: PredictRequest):
    preds = model.predict(req.instances).tolist()
    return {"predictions": preds}
'''
open("/tmp/app.py", "w").write(serving_app)
print("Wrote /tmp/app.py")
print("Run it with:  uvicorn app:app --host 0.0.0.0 --port 8080 --app-dir /tmp")

In [ ]:
# 3) Exercise the prediction logic in-process (what the /predict route does).
import joblib
model = joblib.load("/tmp/iris_model.joblib")
sample = [[5.1, 3.5, 1.4, 0.2], [6.7, 3.0, 5.2, 2.3]]
print("predictions:", model.predict(sample).tolist())

# Equivalent live call once the server is running:
#   curl -s localhost:8080/predict \
#        -H 'content-type: application/json' \
#        -d '{"instances": [[5.1,3.5,1.4,0.2]]}'

## Advanced Features

### Dynamic batching

Under concurrent load, serving each request with its own forward pass wastes the GPU. A model server can hold incoming requests for a few milliseconds and run them as one batch — trading a little latency for a large throughput gain. NVIDIA Triton enables this per-model:

```pbtxt
# config.pbtxt — Triton dynamic batching
max_batch_size: 32
dynamic_batching {
  preferred_batch_size: [8, 16, 32]
  max_queue_delay_microseconds: 2000   # wait up to 2 ms to fill a batch
}
```

### LLM serving with continuous batching

For autoregressive LLMs, classic batching stalls because sequences finish at different steps. **vLLM** uses paged attention + continuous (iteration-level) batching to keep the GPU saturated and exposes an OpenAI-compatible API:

```bash
python -m vllm.entrypoints.openai.api_server \
  --model meta-llama/Llama-3.1-8B-Instruct \
  --tensor-parallel-size 1 --max-model-len 8192
# then POST to http://localhost:8000/v1/chat/completions
```

### Multi-model and ensembles

Triton and KServe can host many models in one process and chain them (an "ensemble"/inference graph): e.g. tokenizer → encoder → classifier, with outputs piped between steps server-side so the client makes one call.

In [ ]:
# Illustration: why batching helps. Per-request fixed overhead amortizes
# across a batch, so effective throughput climbs as batch size grows.
def throughput(reqs_per_batch, fixed_ms=8.0, per_item_ms=1.5):
    batch_latency = fixed_ms + per_item_ms * reqs_per_batch
    return 1000.0 * reqs_per_batch / batch_latency  # requests/sec

for b in (1, 4, 8, 16, 32):
    print(f"batch={b:>2}  ~{throughput(b):6.1f} req/s")

## Use Cases

### Real-world Applications of Model Deployment

#### Use Case 1: Real-time fraud scoring (online)

- **Context** — score every card transaction at checkout within a ~50 ms p99 budget.
- **Implementation** — a low-latency gRPC service backed by a gradient-boosted model; real-time features pulled from a feature store with a Redis cache; autoscaled on QPS.
- **Result** — synchronous decisions in the request path; new model versions canaried at 1% before full promotion.

#### Use Case 2: Nightly churn / propensity scoring (batch)

- **Context** — score tens of millions of customers once a day and write to a warehouse table.
- **Implementation** — a scheduled Spark/Ray or Kubernetes Job that loads the same registered model and writes results to BigQuery/Snowflake; no online server needed.
- **Result** — high throughput, cost-efficient (compute spun up only for the run), results consumed by marketing/CRM systems.

#### Use Case 3: Chatbot / RAG assistant (LLM online)

- **Context** — stream tokens to users with bounded time-to-first-token.
- **Implementation** — vLLM with continuous batching behind a gateway that enforces per-tenant rate limits; streaming responses over SSE; GPU autoscaling on queue depth.
- **Result** — high GPU utilization and predictable latency under bursty chat traffic.

#### Use Case 4: On-device image classification (edge)

- **Context** — classify images on a phone with no network round trip.
- **Implementation** — export to ONNX / TensorFlow Lite, quantize to INT8, run with the on-device runtime.
- **Result** — offline inference, lower latency, and no per-request server cost.

## Best Practices

1. **Pin everything in one artifact.** Package model weights, inference code, preprocessing, and dependency versions together (a container or model archive). The training-time and serving-time transforms must be identical to avoid training/serving skew.
2. **Load the model once.** Initialize weights at process/worker startup, never per request. Use a readiness probe that flips to ready only after the model is loaded.
3. **Separate liveness, readiness, and prediction.** Health probes must not run model inference, or a slow model will fail health checks and cause restart storms.
4. **Version models and APIs explicitly.** Treat a model version like a code release: immutable, traceable to training data/run, and independently rollback-able.
5. **Roll out progressively.** Use shadow → canary → full promotion with automated metric gates (latency, error rate, and a business/quality metric) and instant rollback.
6. **Set resource requests/limits and concurrency.** Right-size CPU/GPU/memory and cap per-replica concurrency so one replica degrades gracefully instead of OOM-killing.
7. **Log predictions asynchronously.** Emit inputs, outputs, and model version to a side channel (not the hot path) for drift detection, debugging, and audit.
8. **Define an explicit latency SLO and load-test to it.** Know your p50/p95/p99 under realistic concurrency before launch, not after an incident.

## Common Pitfalls

1. **Training/serving skew.** Preprocessing differs between training and serving (different library version, missing feature scaling). *Avoid by* sharing one transform implementation and capturing it in the artifact; add a parity test that runs the same input through both paths.
2. **Cold starts.** Loading a multi-GB model on the request path causes timeouts and failed health checks. *Avoid by* loading at startup, keeping a warm pool / min replicas > 0, and using readiness gates so traffic only arrives after warm-up.
3. **Health check runs inference.** A heavyweight `/healthz` makes the pod flap under load. *Avoid by* keeping probes trivial and separate from `/predict`.
4. **Unbounded concurrency / no batching policy.** Either the GPU is idle (no batching) or it OOMs (no max batch / concurrency cap). *Avoid by* tuning batch size, queue delay, and per-replica concurrency together.
5. **No rollback path.** A bad model is promoted to 100% with no way back. *Avoid by* keeping the previous version deployable and gating promotion on metrics.
6. **Latency measured as an average.** A good mean hides a terrible p99. *Avoid by* tracking percentiles and alerting on tail latency.
7. **Silent model drift.** Inputs shift and accuracy decays with no error raised. *Avoid by* logging predictions/features and monitoring distributions against a baseline.

## Performance Optimization

### Optimizing Inference for Production

#### Configuration tuning

- **Batch size & max queue delay** — larger batches raise throughput but add latency; tune `preferred_batch_size` and `max_queue_delay` to your SLO.
- **Replica count & concurrency** — set min replicas to cover baseline traffic; cap per-replica concurrency to avoid memory blowups.
- **Numeric precision** — FP16/BF16 and INT8 quantization cut memory and latency, usually with negligible accuracy loss; validate quality after quantizing.
- **Compiled/optimized runtimes** — TensorRT, ONNX Runtime, `torch.compile`, and vLLM paged attention often give 2–5x speedups over an eager framework.

#### Practical levers

- Use GPU for large deep models; CPU is often cheaper and sufficient for trees/linear models.
- Cache deterministic results and repeated feature lookups.
- Co-locate the feature store/cache with the inference service to cut network latency.

In [ ]:
# Estimate the replica count needed to meet a latency-bounded QPS target.
# A single replica's capacity is roughly: concurrency / service_time.
def replicas_needed(target_qps, p95_latency_s, concurrency_per_replica, headroom=0.7):
    per_replica_qps = concurrency_per_replica / p95_latency_s
    raw = target_qps / (per_replica_qps * headroom)  # leave headroom for spikes
    return max(1, -(-int(raw * 100) // 100))  # ceil-ish, integer replicas

for qps in (100, 500, 2000):
    n = replicas_needed(qps, p95_latency_s=0.040, concurrency_per_replica=8)
    print(f"target {qps:>4} req/s -> {n} replicas")

## Production Deployment

### Containerizing the service

A reproducible image bundles the runtime, dependencies, and serving code. Keep it slim, run as non-root, and expose a health endpoint.

```dockerfile
# Dockerfile
FROM python:3.11-slim
WORKDIR /app

# Install deps first for better layer caching
COPY requirements.txt .
RUN pip install --no-cache-dir -r requirements.txt

# App + model handler (model itself often fetched from the registry at startup)
COPY app.py .

ENV PYTHONUNBUFFERED=1
EXPOSE 8080
# Multiple workers for CPU concurrency; one worker per GPU for GPU models.
CMD ["uvicorn", "app:app", "--host", "0.0.0.0", "--port", "8080", "--workers", "2"]
```

### Kubernetes Deployment

A Deployment runs N replicas; a Service load-balances them; probes gate traffic; an HPA autoscales on load.

```yaml
apiVersion: apps/v1
kind: Deployment
metadata:
  name: iris-classifier
spec:
  replicas: 3
  selector:
    matchLabels: { app: iris-classifier }
  template:
    metadata:
      labels: { app: iris-classifier, version: v1 }
    spec:
      containers:
        - name: server
          image: registry.example.com/iris-classifier:1.0.0
          ports: [{ containerPort: 8080 }]
          resources:
            requests: { cpu: "500m", memory: "512Mi" }
            limits:   { cpu: "1",    memory: "1Gi" }
          readinessProbe:                    # only receive traffic once model is loaded
            httpGet: { path: /healthz, port: 8080 }
            initialDelaySeconds: 10
            periodSeconds: 5
          livenessProbe:
            httpGet: { path: /healthz, port: 8080 }
            initialDelaySeconds: 20
            periodSeconds: 10
---
apiVersion: v1
kind: Service
metadata:
  name: iris-classifier
spec:
  selector: { app: iris-classifier }
  ports: [{ port: 80, targetPort: 8080 }]
---
apiVersion: autoscaling/v2
kind: HorizontalPodAutoscaler
metadata:
  name: iris-classifier
spec:
  scaleTargetRef:
    apiVersion: apps/v1
    kind: Deployment
    name: iris-classifier
  minReplicas: 3
  maxReplicas: 20
  metrics:
    - type: Resource
      resource:
        name: cpu
        target: { type: Utilization, averageUtilization: 60 }
```

### Rollout strategies

- **Rolling update** (default) — replace pods incrementally; simple, brief version mix.
- **Blue-green** — stand up the new version in full, flip the Service selector, keep blue for instant rollback.
- **Canary** — route a small % to the new version (via Istio/Argo Rollouts), watch metrics, then ramp.
- **Shadow** — mirror live traffic to the candidate without returning its responses, to compare quality risk-free.

A higher-level option is **KServe**, which wraps all of this in an `InferenceService` CRD with built-in canary, scale-to-zero, and multi-framework runtimes.

## Monitoring and Observability

### What to watch in a deployed model

Operational health follows the **four golden signals**, plus ML-specific quality signals.

#### Key metrics to track

- **Latency** — request duration p50/p95/p99 (alert on the tail, not the mean).
- **Traffic** — requests per second per model version.
- **Errors** — 5xx rate, validation rejects, inference exceptions.
- **Saturation** — CPU/GPU utilization, GPU memory, queue depth, batch fill rate.
- **Model quality / drift** — input feature distributions vs. a baseline, prediction distribution, and (when labels arrive) live accuracy/AUC.

```python
# Prometheus instrumentation sketch for the FastAPI app
from prometheus_client import Counter, Histogram
PRED = Counter("predictions_total", "predictions", ["model_version", "status"])
LAT  = Histogram("predict_latency_seconds", "predict latency",
                 buckets=(0.005, 0.01, 0.025, 0.05, 0.1, 0.25, 0.5, 1.0))
# wrap the /predict handler:  with LAT.time(): preds = model.predict(...)
#                             PRED.labels(MODEL_VERSION, "ok").inc()
```

#### Logging best practices

- Emit **structured JSON logs** (one event per request) with a request id, model version, and latency — never log raw PII.
- Write **prediction logs asynchronously** to S3/BigQuery for drift analysis and audit; keep them off the hot path.
- Use **OpenTelemetry traces** to follow a request across gateway → feature store → model.
- Trigger **drift/quality alerts** (e.g. PSI or KS test over a window) so you retrain before users notice degradation.

## Troubleshooting

#### Issue 1: High tail latency (p99 spikes) under load

**Symptoms** — mean latency is fine, but p99 occasionally jumps 5–10x; sporadic timeouts.

**Cause** — cold starts, GC/allocation pauses, no batching saturating the GPU, or queueing because concurrency exceeds capacity.

**Solution** — keep min replicas > 0 with warm models, tune dynamic batching + per-replica concurrency, profile to find the stall, and scale out on a latency or queue-depth metric rather than CPU alone.

#### Issue 2: Predictions differ between offline and online

**Symptoms** — the same input yields different results in the notebook vs. the deployed service.

**Cause** — training/serving skew: divergent preprocessing, library versions, or feature values fetched at serving time.

**Solution** — share one transform implementation, pin dependency versions in the image, and add a parity test asserting identical outputs for a fixed input set across both paths.

#### Issue 3: Pods crash-looping or OOM-killed

**Symptoms** — `CrashLoopBackOff`, restarts, `OOMKilled` in pod status.

**Cause** — memory limit too low for the loaded model, unbounded request concurrency, or the model loading inside the liveness probe.

**Solution** — raise memory requests/limits to fit the model plus batch, cap concurrency, and ensure health probes do no inference; check `kubectl describe pod` and container logs for the exact failure.

#### Issue 4: New model version regresses quality after rollout

**Symptoms** — error rate or a quality metric degrades after a deploy.

**Cause** — the candidate was promoted without a gated comparison against production traffic.

**Solution** — roll back instantly to the previous version (keep it deployable), then re-evaluate the candidate via shadow/canary with automated metric gates before re-promoting.

## Comparison with Alternatives

### Serving approaches compared

| Aspect | Custom FastAPI/Flask | Dedicated model server (Triton / TorchServe / KServe) | Managed endpoint (SageMaker / Vertex) |
|--------|----------------------|-------------------------------------------------------|----------------------------------------|
| Setup effort | Low to start | Medium | Lowest (fully managed) |
| Multi-framework | Manual | Built-in | Built-in |
| Dynamic batching | Hand-rolled | Built-in | Built-in |
| Autoscaling / scale-to-zero | DIY (HPA/KEDA) | KServe: built-in | Built-in |
| GPU optimization (TensorRT, etc.) | Manual | First-class | Managed |
| Operational control | Full | Full | Limited (vendor) |
| Cost model | Your infra | Your infra | Pay-per-use + premium |
| Best for | Simple/CPU models, full control | Standardized high-throughput serving | Teams wanting minimal ops |

For **LLMs specifically**, prefer a purpose-built server: **vLLM** or **TGI** for continuous batching and paged attention, vs. a generic server that lacks these optimizations.

### When to choose what

- **Custom server** — a single simple model, CPU-bound, or you need full control and minimal moving parts.
- **Dedicated model server** — many models/frameworks, high throughput, GPU optimization, and standardized ops across teams.
- **Managed endpoint** — small ops team, want autoscaling/monitoring out of the box, and accept vendor lock-in and premium cost.
- **Edge runtime (ONNX Runtime / TF Lite / Core ML)** — inference must run on-device or offline.

## Resources

### Documentation

- KServe — model inference on Kubernetes: https://kserve.github.io/website/
- NVIDIA Triton Inference Server: https://docs.nvidia.com/deeplearning/triton-inference-server/
- TorchServe: https://pytorch.org/serve/
- vLLM (LLM serving): https://docs.vllm.ai/
- BentoML (framework-agnostic packaging): https://docs.bentoml.org/
- Seldon Core: https://docs.seldon.io/
- ONNX Runtime: https://onnxruntime.ai/docs/

### Cloud managed serving

- AWS SageMaker Inference: https://docs.aws.amazon.com/sagemaker/latest/dg/deploy-model.html
- Google Vertex AI Prediction: https://cloud.google.com/vertex-ai/docs/predictions/overview
- Azure ML Online Endpoints: https://learn.microsoft.com/azure/machine-learning/concept-endpoints

### Background reading

- Google SRE Book — Monitoring Distributed Systems (four golden signals): https://sre.google/sre-book/monitoring-distributed-systems/
- "Hidden Technical Debt in Machine Learning Systems" (Sculley et al., 2015): https://papers.nips.cc/paper/5656-hidden-technical-debt-in-machine-learning-systems
- Argo Rollouts (canary/blue-green for Kubernetes): https://argoproj.github.io/argo-rollouts/

### Related techniques

- Quantization and pruning (smaller, faster models for serving)
- Feature stores (consistent online/offline features)
- CI/CD for ML and model registries (automated promotion pipelines)